In [ ]:
import os
import requests
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# If you installed PyTorch to bypass the Windows TF bug, uncomment this:
os.environ["KERAS_BACKEND"] = "torch" 

import keras
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout, BatchNormalization
from keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler

# 1. Verify GPU Availability
print(f"Using Keras backend: {keras.backend.backend()}")
import tensorflow as tf
print("GPU Active and Ready: ", len(tf.config.list_physical_devices('GPU')) > 0)

# 2. Automatically Create the Data Storing Folders (Relative to root/notebooks)
os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)
print("Target directories verified/created in the root folder!")

Using Keras backend: tensorflow


In [ ]:
# ─── REGIONAL CONFIGURATION ───
CITIES = {
    'Jakarta':   {'lat': -6.2088, 'lon': 106.8456},
    'Bangkok':   {'lat': 13.7563, 'lon': 100.5018},
    'Manila':    {'lat': 14.5995, 'lon': 120.9842},
    'Singapore': {'lat': 1.3521,  'lon': 103.8198}
}

START_DATE = '2010-01-01'
END_DATE = '2024-12-31'
WINDOW_SIZE = 14
FEATURES = ['T_max', 'RH_max', 'Wind', 'Heat_Index', 'doy_sin', 'doy_cos']

def calculate_apparent_temperature(t, rh, ws):
    """Australian AT formula (incorporates wind cooling)"""
    e = (rh / 100.0) * 6.105 * np.exp((17.27 * t) / (237.7 + t))
    return t + (0.33 * e) - (0.70 * (ws / 3.6)) - 4.00

def create_sequences(data, target, window):
    X, y = [], []
    for i in range(len(data) - window):
        X.append(data[i : i + window])
        y.append(target[i + window])
    return np.array(X), np.array(y)

# ─── START THE REVENUE PIPELINE ───
for city, coords in CITIES.items():
    print(f"\n{'='*50}\n🛰️ PROTRACTED OPERATION: {city.upper()}\n{'='*50}")
    
    # 1. Fetch from Open-Meteo API
    url = (
        f"https://archive-api.open-meteo.com/v1/archive?latitude={coords['lat']}&longitude={coords['lon']}"
        f"&start_date={START_DATE}&end_date={END_DATE}"
        f"&daily=temperature_2m_max,relative_humidity_2m_max,windspeed_10m_max&timezone=Asia%2FSingapore"
    )
    response = requests.get(url).json()
    df = pd.DataFrame(response['daily'])
    df['time'] = pd.to_datetime(df['time'])
    df.set_index('time', inplace=True)
    df.columns = ['T_max', 'RH_max', 'Wind']
    df = df.ffill().bfill()
    
    # 💾 DATA STORING: Dump unmodified baseline down to raw folder
    df.to_csv(f'../data/raw/{city.lower()}_raw.csv')
    print(f"  -> Saved Raw: data/raw/{city.lower()}_raw.csv")

    # 2. Climate Math & Feature Engineering
    df['Heat_Index'] = calculate_apparent_temperature(df['T_max'], df['RH_max'], df['Wind'])
    df['doy_sin'] = np.sin(2 * np.pi * df.index.dayofyear / 365)
    df['doy_cos'] = np.cos(2 * np.pi * df.index.dayofyear / 365)
    
    # Calculate Localized 95th Percentile Limit
    threshold_95 = df['Heat_Index'].quantile(0.95)
    df['Target'] = (df['Heat_Index'] >= threshold_95).astype(int)
    
    # 💾 DATA STORING: Dump engineered features to processed folder
    df.to_csv(f'../data/processed/{city.lower()}_processed.csv')
    print(f"  -> Saved Processed: data/processed/{city.lower()}_processed.csv")

    # 3. Time Series Partitioning
    train_df = df[df.index < '2022-01-01']
    
    # Scale features relative strictly to the training split baseline
    scaler = MinMaxScaler()
    train_scaled = scaler.fit_transform(train_df[FEATURES])
    
    # Save mathematical artifacts for Streamlit lookup
    joblib.dump(scaler, f'../models/scaler_{city.lower()}.pkl')
    joblib.dump(threshold_95, f'../models/threshold_{city.lower()}.pkl')

    X_train, y_train = create_sequences(train_scaled, train_df['Target'].values, WINDOW_SIZE)

    # 4. Neural Net Construction
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(WINDOW_SIZE, len(FEATURES))),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        BatchNormalization(),
        Dense(16, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001), 
        loss='binary_crossentropy', 
        metrics=[keras.metrics.AUC(name='auc')]
    )
    
    # Fire up training with the 15x penalty weight engine active
    early_stop = EarlyStopping(monitor='val_auc', patience=8, mode='max', restore_best_weights=True)
    
    print(f"  🧠 Training LSTM (Threshold: {threshold_95:.2f}°C)...")
    model.fit(
        X_train, y_train, validation_split=0.15, epochs=50, batch_size=32,
        class_weight={0: 1.0, 1: 15.0}, callbacks=[early_stop], verbose=0
    )
    
    # Save final model file
    model.save(f'../models/{city.lower()}_lstm.keras')
    print(f"  🎯 Completed! Weights saved to models/{city.lower()}_lstm.keras")

print("\n🎉 PIPELINE EXECUTION REVENUE COMPLETE! All data stored and models baked.")

Fetching data from Open-Meteo for 2010-01-01 to 2024-12-31...

🔥 95th Percentile Threshold for Jakarta: 42.67°C
Normal Days: 5205 | Heatwave Days: 274


,T_max,RH_max,Precip,Wind,Heat_Index,doy_sin,doy_cos,Target
time,,,,,,,,
2024-12-27,32.1,89,1.3,9.7,40.208115,-5.161967e-02,0.998667,0
2024-12-28,33.2,86,0.2,12.8,41.095419,-3.442161e-02,0.999407,0
2024-12-29,32.0,92,1.9,10.1,40.420672,-1.721336e-02,0.999852,0
2024-12-30,30.2,88,4.2,14.6,35.782694,6.432491e-16,1.000000,0
2024-12-31,31.4,87,1.2,14.9,37.651672,1.721336e-02,0.999852,0


In [12]:
# ─── TRAIN / TEST SPLIT ───
split_date = '2022-01-01'
train_df = df[df.index < split_date]
test_df = df[df.index >= split_date]

# The finalized list of features!
features = ['T_max', 'RH_max', 'Wind', 'Heat_Index', 'doy_sin', 'doy_cos']

# ─── SCALING ───
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_df[features])
test_scaled = scaler.transform(test_df[features]) 

# Save the scaler directly to your models folder
os.makedirs('../models', exist_ok=True)
joblib.dump(scaler, '../models/scaler_jakarta.pkl')
joblib.dump(threshold_95, '../models/threshold_jakarta.pkl') # Save threshold for the app!
print("✅ Scaler and Threshold saved to '../models/'")

# ─── SEQUENCE GENERATOR ───
def create_sequences(feature_data, target_data, window):
    X, y = [], []
    for i in range(len(feature_data) - window):
        X.append(feature_data[i : i + window])
        y.append(target_data[i + window])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_scaled, train_df['Target'].values, WINDOW_SIZE)
X_test, y_test = create_sequences(test_scaled, test_df['Target'].values, WINDOW_SIZE)
print(f"X_train shape: {X_train.shape}")

✅ Scaler and Threshold saved to '../models/'
X_train shape: (4369, 14, 6)


In [13]:
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(WINDOW_SIZE, len(features))),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    BatchNormalization(),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid') # Outputs a probability from 0.0 to 1.0!
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001), 
    loss='binary_crossentropy', 
    metrics=[keras.metrics.AUC(name='auc')]
)

# The AI gets punished 15x harder for missing a heatwave
weights = {0: 1.0, 1: 15.0}

early_stop = EarlyStopping(monitor='val_auc', patience=10, mode='max', restore_best_weights=True)

print("🚀 Training the Model...")
history = model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=50,
    batch_size=32,
    class_weight=weights, 
    callbacks=[early_stop],
    verbose=1
)

model.save('../models/jakarta_lstm.keras')
print("✅ Training complete. Model saved to '../models/jakarta_lstm.keras'")

🚀 Training the Model...
Epoch 1/50


d:\Project\heatwave-predictor\venv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


117/117 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - auc: 0.8642 - loss: 0.4495 - val_auc: 0.8121 - val_loss: 0.3647
Epoch 2/50
117/117 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - auc: 0.9112 - loss: 0.3578 - val_auc: 0.7374 - val_loss: 0.2255
Epoch 3/50
117/117 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - auc: 0.9316 - loss: 0.3108 - val_auc: 0.7039 - val_loss: 0.2236
Epoch 4/50
117/117 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - auc: 0.9434 - loss: 0.2844 - val_auc: 0.7370 - val_loss: 0.1749
Epoch 5/50
117/117 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - auc: 0.9434 - loss: 0.2830 - val_auc: 0.7589 - val_loss: 0.1844
Epoch 6/50
117/117 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - auc: 0.9388 - loss: 0.2858 - val_auc: 0.8609 - val_loss: 0.2122
Epoch 7/50
117/117 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - auc: 0.9482 - loss: 0.2649 - val_auc: 0.8588 - val_loss: 0.1854
Epoch 8/50
117/117 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - auc: 0.9485 - loss: 0.2662 - val_auc: 0.8803 - val_loss: 0.2901
Epoch 9/50
117/117 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - auc: 0.9492 

In [15]:
# 1. Fetch fresh data (2025 to Mid 2026)
df_future = fetch_weather_data(LAT, LON, '2025-01-01', '2026-05-10')

# 2. Apply Math & Calendar
df_future['Heat_Index'] = calculate_apparent_temperature(df_future['T_max'], df_future['RH_max'], df_future['Wind'])
df_future['doy_sin'] = np.sin(2 * np.pi * df_future.index.dayofyear / 365)
df_future['doy_cos'] = np.cos(2 * np.pi * df_future.index.dayofyear / 365)

# 3. Apply the Historical Threshold Target
df_future['Target'] = (df_future['Heat_Index'] >= threshold_95).astype(int)

# 4. Load Scaler & Model
scaler_loaded = joblib.load('../models/scaler_jakarta.pkl')
model_loaded = keras.models.load_model('../models/jakarta_lstm.keras')

# 5. Scale & Sequence
future_scaled = scaler_loaded.transform(df_future[features])
X_future, y_future_actual = create_sequences(future_scaled, df_future['Target'].values, WINDOW_SIZE)

# 6. Predict!
y_future_pred = model_loaded.predict(X_future).flatten()

# 7. Print the results
plot_dates = df_future.index[WINDOW_SIZE:]
results_df = pd.DataFrame({
    'Actual_Heatwave': y_future_actual,
    'AI_Risk_Score': y_future_pred
}, index=plot_dates)

print("\n🔥 TOP 10 HIGHEST RISK DAYS PREDICTED BY AI:")
display(results_df.sort_values(by='AI_Risk_Score', ascending=False).head(10).round(4))

print("\n📅 HOW IT PERFORMED LAST WEEK (May 2026):")
display(results_df.tail(7).round(4))

Fetching data from Open-Meteo for 2025-01-01 to 2026-05-10...
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step

🔥 TOP 10 HIGHEST RISK DAYS PREDICTED BY AI:


,Actual_Heatwave,AI_Risk_Score
time,,
2025-10-22,1,0.8826
2025-10-24,0,0.8804
2025-10-23,0,0.8803
2025-10-21,0,0.8778
2025-10-20,0,0.8701
2025-10-25,0,0.8564
2025-10-19,0,0.8496
2025-10-18,0,0.8311
2025-09-09,0,0.8221



📅 HOW IT PERFORMED LAST WEEK (May 2026):


,Actual_Heatwave,AI_Risk_Score
time,,
2026-05-04,0,0.1807
2026-05-05,0,0.1602
2026-05-06,0,0.1493
2026-05-07,0,0.1404
2026-05-08,0,0.1344
2026-05-09,0,0.1092
2026-05-10,1,0.0907
